In [1]:
import numpy as np
import xtrack as xt
import json
import sys
helpers_path = f'../' # specify the path to the helper_functions folder
sys.path.insert(0, helpers_path)
from helpers_for_imperfections_model import compute_pseudo_inverse, apply_optics_correction
from helpers_from_xutil import add_chroma_knobs, match_tune_chroma

In [2]:
# CHOOSE SEED HERE
seed = 4

In [4]:
line_version = "LCC_106-2-3_z"

# Load reference line
line0 = xt.Line.from_json('lattices/reference_lattice_LCC_V106/line_fccee_p_ring_LCC_106-2-3_z_merged_dipoles_with_correctors.json')
line0.cycle(name_first_element='rf400', inplace=True) # cycle to rf cavity
line0.configure_radiation(model=None, model_beamstrahlung=None) # disable radiation
line0.twiss_default['method'] = '4d' # switch to 4d Twiss method

tw_ref = line0.twiss(coupling_edw_teng=True, matrix_stability_tol=0.20)
for col in ['f1001', 'f1010']:
    tw_ref[col + "c"] = np.real(tw_ref[col])
    tw_ref[col + "s"] = np.imag(tw_ref[col])

Loading line from dict: 100%|██████████| 15545/15545 [00:03<00:00, 4891.09it/s]


Done loading line from dict.           


In [5]:
# Load orbit-corrected line (cycled to rf cavity, radiation disabled, 4d method)
line = xt.Line.from_json(f'lattices/lattices_with_corrected_imperfections/01_orbit_corrected_only/{line_version}_line_orbit_corrected_seed{seed}.json')
# just to be safe: 
line.cycle(name_first_element='rf400', inplace=True) # cycle to rf cavity
line.configure_radiation(model=None, model_beamstrahlung=None) #disable radiation
line.twiss_default['method'] = '4d' # switch to 4d Twiss method

Loading line from dict: 100%|██████████| 15545/15545 [00:03<00:00, 4842.18it/s]


Done loading line from dict.           


### Get observation points, observables, response matrix inverses

In [4]:
# Define observation points and observables
OBSERVATION_POINTS = json.load(open(f'helpers_for_RM_generation/observation_points.json', 'r'))
OBSERVABLES_NORMAL = ['mux', 'muy', 'dx']
OBSERVABLES_SKEW = ['f1001s', 'f1001c', 'f1010s', 'f1010c', 'dy']

In [5]:
# Choose the threshold for singular values when computing the pseudo-inverse of the response matrix
epsilon = 0.001
# Choose the number of iterations for the optics correction
nloops = 5

In [6]:
# Normal correctors

# Load response matrix for normal correctors
corr_type = 'knl1'
RM_normal = np.load(f'helpers_for_RM_generation/RM_{corr_type}_{"_".join(OBSERVABLES_NORMAL)}.npy')
corr_knob_names_normal = json.load(open(f'helpers_for_RM_generation/corr_knob_names_{corr_type}_{"_".join(OBSERVABLES_NORMAL)}.json'))

# Invert RM
RM_inv_normal, U, S_normal, VT = compute_pseudo_inverse(RM_normal, full_matrices=False, epsilon=epsilon)

In [7]:
# Skew correctors

# Load response matrix for skew correctors
corr_type = 'ksl1'
RM_skew = np.load(f'helpers_for_RM_generation/RM_{corr_type}_{"_".join(OBSERVABLES_SKEW)}.npy')
corr_knob_names_skew = json.load(open(f'helpers_for_RM_generation/corr_knob_names_{corr_type}_{"_".join(OBSERVABLES_SKEW)}.json'))

# Invert RM
RM_inv_skew, U, S_skew, VT = compute_pseudo_inverse(RM_skew, full_matrices=False, epsilon=epsilon)

### Apply optics corrections

In [14]:
# Apply the optics corrections
    
add_chroma_knobs(line, optics_type='LCC')

for _ in range(2):  # Repeat the process twice for better convergence
    #  NORMAL
    apply_optics_correction(line, RM_inv_normal, corr_knob_names_normal, tw_ref=tw_ref, OBSERVATION_POINTS=OBSERVATION_POINTS, OBSERVABLES=OBSERVABLES_NORMAL, nloops=nloops)
    print('Applied optics correction with normal correctors.')

    # Again, as in the orbit correction, it is easier to use try/except for tune matching 
    # as the optics may still be so distorted during the first few iterations that tune matchin gmay not be immediately possible
    try: 
        match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
        print('Tune matched successfully.')
        
        match_tune_chroma(line, tw_ref, match_quantities='chroma', method='4d')
        print('Chroma matched successfully.')
    except Exception as e:
        print(f'Tune/chroma matching failed due to: {e}')

    # SKEW
    apply_optics_correction(line, RM_inv_skew, corr_knob_names_skew, tw_ref=tw_ref, OBSERVATION_POINTS=OBSERVATION_POINTS, OBSERVABLES=OBSERVABLES_SKEW, nloops=nloops)
    print('Applied optics correction with skew correctors.')

    try: 
        match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
        print('Tune matched successfully.')
        
        match_tune_chroma(line, tw_ref, match_quantities='chroma', method='4d')
        print('Chroma matched successfully.')
    except Exception as e:
        print(f'Tune/chroma matching failed due to: {e}')

Applied optics correction with normal correctors.
                                             
Optimize - start penalty: 0.3285                            
Matching: model call n. 11 penalty = 1.9117e-07              
Optimize - end penalty:  1.9117e-07                            
Target status:               nalty = 1.9117e-07              
id state tag  tol_met       residue   current_val    target_val description                           
0  ON    tune    True   1.35715e-08        194.16        194.16 'qx', val=194.16, tol=1e-05, weight=10
1  ON    tune    True  -1.34639e-08         170.2         170.2 'qy', val=170.2, tol=1e-05, weight=10 
Vary status:                 
id state tag  met name lower_limit   current_val upper_limit val_at_iter_0          step        weight
0  ON    quad OK  kqf2 None           0.00926683 None           0.00926769         1e-08             1
1  ON    quad OK  kqd1 None           -0.0136889 None            -0.013691         1e-08             1
Tune ma

In [15]:
# # Save the optics-corrected line
# line.to_json(f'lattices/lattices_with_corrected_imperfections/02_orbit_and_optics_corrected/{line_version}_line_optics_corrected_seed{seed}_without_radiation.json')

# Enable radiation and tapering and switch to 6d Twiss method
line.twiss_default['method'] = '6d'
line.configure_radiation(model='mean', model_beamstrahlung=None)
line.compensate_radiation_energy_loss()

# Save the optics-corrected line with radiation
line.to_json(f'lattices/lattices_with_corrected_imperfections/02_orbit_and_optics_corrected/{line_version}_line_optics_corrected_seed{seed}.json')

Compensating energy loss.
Share energy loss among cavities (repeat until energy loss is zero)
Energy loss: 34_787_029.953 eV             
Energy loss: 106_151.838 eV             
Energy loss: 324.290 eV             
Energy loss: 0.992 eV             
Energy loss: -53_917.752 eV             
Energy loss: -246.805 eV             
Energy loss: 163.401 eV             
Energy loss: 1.498 eV             

  - Set delta_taper
  - Restore cavity voltage and frequency. Set cavity lag
